# 04 — Avaliação da recuperação e do RAG

## Objetivo
Avaliar a recuperação semântica com um conjunto de perguntas anotadas, inspecionar evidências e verificar os cenários de resposta sem evidência e fora do escopo.

Este notebook mede o retrieval bruto com Precision@K, Recall@K e MRR. A avaliação quantitativa depende da qualidade e abrangência das anotações em `data/evaluation/retrieval_gold.json`; os resultados não devem ser interpretados como uma medida universal se o conjunto for pequeno.

A avaliação de recuperação não exige executar o LLM. O foco aqui é verificar se os documentos relevantes aparecem entre os candidatos recuperados.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
import json
import pandas as pd

from config.settings import settings
from src.embeddings.embedder import EmbeddingModel
from src.retrieval.retriever import VectorRetriever
from src.rag.pipeline import RAGPipeline
from src.evaluation.evaluator import evaluate_queries

INDEX_DIR = settings.data_dir / "vectorstore"
GOLD_FILE = ROOT / "data" / "evaluation" / "retrieval_gold.json"

retriever = VectorRetriever.load(INDEX_DIR)
embedder = EmbeddingModel(settings.embedding_model)
pipeline = RAGPipeline(
    embedder=embedder,
    retriever=retriever,
    generator=None,
    top_k=settings.top_k,
    min_relevance_score=settings.min_relevance_score,
    min_scope_score=settings.min_scope_score,
    min_domain_score=settings.min_domain_score,
)

print("Vetores no índice:", retriever.size)
print("Modelo:", settings.embedding_model)


## 1. Conjunto de avaliação

Cada pergunta deve ter IDs de avaliações consideradas relevantes após inspeção manual. Não derive relevância apenas dos documentos que o próprio sistema retornou: isso introduz viés na avaliação.

In [ ]:
with GOLD_FILE.open(encoding="utf-8") as file:
    gold_queries = json.load(file)

print("Perguntas anotadas:", len(gold_queries))
for item in gold_queries:
    print(f"- {item['question']} | IDs relevantes: {len(item['relevant_ids'])}")


## 2. Métricas de recuperação

- **Precision@K:** proporção dos K primeiros resultados que são relevantes.
- **Recall@K:** proporção dos documentos relevantes recuperados entre os K primeiros.
- **MRR:** inverso da posição do primeiro documento relevante, considerando zero quando não há acerto.

A função de avaliação é reutilizada de `src/evaluation/evaluator.py`.

In [ ]:
rows = evaluate_queries(pipeline, gold_queries, k=settings.top_k)
metrics = pd.DataFrame(rows)
display(metrics[["question", "precision_at_k", "recall_at_k", "mrr"]])

summary = pd.DataFrame([{
    f"Precision@{settings.top_k}": metrics["precision_at_k"].mean(),
    f"Recall@{settings.top_k}": metrics["recall_at_k"].mean(),
    "MRR": metrics["mrr"].mean(),
}])
display(summary)


## 3. Auditoria dos resultados

Inspecione os IDs recuperados por pergunta. Compare cada resultado com os comentários originais e revise as anotações quando necessário. Um documento relevante não incluído no conjunto anotado será contabilizado como não relevante, mesmo que seja semanticamente adequado.

In [ ]:
for row in rows:
    print("\nPERGUNTA:", row["question"])
    print("IDs relevantes anotados:", next(x["relevant_ids"] for x in gold_queries if x["question"] == row["question"]))
    print("IDs recuperados:", row["retrieved_ids"])
    print(
        f"P@{settings.top_k}={row['precision_at_k']:.3f} | "
        f"R@{settings.top_k}={row['recall_at_k']:.3f} | "
        f"MRR={row['mrr']:.3f}"
    )


## 4. Inspeção qualitativa de uma consulta

Além das métricas, verifique se o texto recuperado realmente responde à pergunta. Scores altos não garantem, isoladamente, relevância factual.

In [ ]:
inspection_question = "Quais são os principais problemas relacionados à entrega?"
candidates = pipeline.retrieve_candidates(inspection_question)

for item in candidates:
    print(f"\n[{item['document_id']}] score={item['score']:.4f}")
    print("Nota:", item.get("metadata", {}).get("review_score"))
    print(item["text"])


## 5. Cenário sem evidência suficiente

O limiar de relevância é elevado deliberadamente para testar o caminho de segurança sem depender de uma pergunta específica.

In [ ]:
strict_pipeline = RAGPipeline(
    embedder=embedder,
    retriever=retriever,
    generator=None,
    top_k=settings.top_k,
    min_relevance_score=0.99,
    min_scope_score=settings.min_scope_score,
    min_domain_score=settings.min_domain_score,
)
no_evidence = strict_pipeline.retrieve_candidates(inspection_question)
print("Candidatos acima do limiar:", len(no_evidence))
assert len(no_evidence) == 0


## 6. Cenário fora do domínio

O teste de domínio pode ser inspecionado diretamente pelo score das âncoras semânticas, sem chamar o gerador.

In [ ]:
out_of_domain_question = "Qual é a previsão do tempo para amanhã?"
print("Pergunta:", out_of_domain_question)
print("Score de domínio:", round(pipeline.domain_score(out_of_domain_question), 4))
print("Limite configurado:", settings.min_domain_score)


## 7. Exportação dos resultados

Os resultados são gravados em `data/evaluation/retrieval_results_notebook.json` para permitir comparação entre execuções. O arquivo gerado não substitui a revisão manual das anotações.

In [ ]:
output_file = ROOT / "data" / "evaluation" / "retrieval_results_notebook.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
output_file.write_text(
    json.dumps({"summary": summary.to_dict(orient="records")[0], "details": rows}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Resultados salvos em:", output_file.relative_to(ROOT))


## 8. Conclusões da avaliação

Registre os resultados observados, limitações do conjunto anotado, exemplos de acertos/erros e oportunidades de melhoria. Não ajuste as anotações apenas para elevar as métricas: relevância deve ser confirmada pela leitura dos comentários.

**Notebook de ponta a ponta:** [00 — RAG completo](00_rag_completo.ipynb).